# Phase 1.2: classifier v2 with a 7th class `not_steel` (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajaybahuguna17/qc-defect-detector/blob/feature/not-steel-class/phase1-vision/notebooks/train_colab_phase1_2.ipynb)

Goal: photos that are not a steel surface (people, fields, ground, walls, boards, objects) are
**rejected** with decision `RETAKE` instead of getting a defect class. The **detector is unchanged**
(the Phase 1.1 detector of run `p11_split701515_det320_cls224_s42`, read from Drive, read only).

1. clone at `BRANCH`, pinned install
2. steel data: the **same** NEU 70/15/15 split as Phase 1.1 (seed 42), asserted identical to
   `results/phase1_2/neu_split_manifest_s42.json`
3. `not_steel` data (~2000): COCO val2017 (1000), DTD (600, all 47 texture categories), Places365
   val_256 (400, fields / ground / roads / buildings), from the official URLs (`DATA_SOURCES.md`);
   ~50 % grayscale, ~50 % random-crop resized to 200x200; split 70/15/15 (seed 42), zero overlap
4. classifier: YOLOv8n-cls from `yolov8n-cls.pt`, 7 classes, imgsz 224, patience 20, seed 42;
   outputs reordered to the contract order (6 NEU classes, then `not_steel`)
5. export FP32 TFLite (`export_cls_tflite.py --num-classes 7`, output asserted `[1, 7]`;
   INT8 not attempted, recorded as skipped)
6. thresholds on **VAL only**: `t_ns` (reject if top-1 is not_steel and p >= t_ns, steel false
   rejection <= 1 %), re-check the classifier threshold (Phase 1.1: 0.64)
7. report on **TEST only**: 7x7 confusion, steel fused accuracy with the Phase 1.1 detector,
   not_steel rejection rate, steel false-rejection rate
8. save to the Drive run folder `p12_cls7_notsteel_s42` (the Phase 1 and Phase 1.1 folders are
   only read and are checked unchanged)

## 0. Configuration

In [ ]:
# ---- edit here --------------------------------------------------------------
REPO_URL     = "https://github.com/ajaybahuguna17/qc-defect-detector.git"
BRANCH       = "feature/not-steel-class"
SEED         = 42
SPLIT        = "70/15/15"
CLS_EPOCHS, CLS_PATIENCE, CLS_IMGSZ, CLS_BATCH = 100, 20, 224, 64
DET_IMGSZ    = 320                 # Phase 1.1 detector input (unchanged)
BASE_CONF, NMS_IOU = 0.05, 0.45    # detector decode, same as the Phase 1.1 evaluation
EXPORT_TIMEOUT_MIN = 15
MAX_FALSE_REJECT   = 0.01          # t_ns: steel images rejected as not_steel on VAL <= 1 %
N_COCO, N_DTD, N_PLACES = 1000, 600, 400
GRAY_FRAC, CROP_FRAC    = 0.5, 0.5
USE_DRIVE    = True                # required: the Phase 1.1 detector is read from Drive
DRIVE_ROOT   = "/content/drive/MyDrive/qc-defect-detector/phase1"
OLD_RUN_NAME = "neu_yolov8n_320_e100_s42"            # Phase 1: read only
P11_RUN_NAME = "p11_split701515_det320_cls224_s42"   # Phase 1.1: read only (detector source)
RUN_NAME     = "p12_cls7_notsteel_s42"
ALLOW_CPU    = False
# ------------------------------------------------------------------------------
assert RUN_NAME not in (OLD_RUN_NAME, P11_RUN_NAME)
assert USE_DRIVE, "the Phase 1.1 detector is read from Drive"
import os, sys
from pathlib import Path
REPO_DIR    = Path("/content/qc-defect-detector")
PHASE1      = REPO_DIR / "phase1-vision"
WORK        = Path("/content/work12")
NS_DL       = Path("/content/ns_download")      # downloaded archives + extracted sources (untrusted data)
EXPORT_CLS  = WORK / "export" / "classifier"
RESULTS_DIR = WORK / "results"
for d in (WORK, NS_DL, EXPORT_CLS, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
CLS_RUN = f"{RUN_NAME}_cls"
print("run name:", RUN_NAME, "| branch:", BRANCH)

## 1. GPU check, Google Drive, fingerprint the Phase 1 and Phase 1.1 folders (read only)

In [ ]:
import subprocess, torch, hashlib
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU, then Run all again (or set ALLOW_CPU=True).")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

def tree_fingerprint(root):
    root = Path(root)
    if not root.exists():
        return None
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(root.rglob("*")) if p.is_file()}

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DEST = Path(DRIVE_ROOT) / RUN_NAME
OLD_DRIVE  = Path(DRIVE_ROOT) / OLD_RUN_NAME
P11_DRIVE  = Path(DRIVE_ROOT) / P11_RUN_NAME
if DRIVE_DEST.exists():
    raise FileExistsError(f"{DRIVE_DEST} already exists. Change RUN_NAME (or delete that folder yourself).")
if not (P11_DRIVE / "models" / "detector").is_dir():
    raise FileNotFoundError(f"Phase 1.1 detector folder not found: {P11_DRIVE / 'models' / 'detector'}")
OLD_FINGERPRINT = tree_fingerprint(OLD_DRIVE)
P11_FINGERPRINT = tree_fingerprint(P11_DRIVE)
print("Drive output:", DRIVE_DEST)
print("Phase 1 folder:", OLD_DRIVE, "->", "missing" if OLD_FINGERPRINT is None else f"{len(OLD_FINGERPRINT)} files fingerprinted")
print("Phase 1.1 folder:", P11_DRIVE, "->", f"{len(P11_FINGERPRINT)} files fingerprinted (read only)")

## 2. Clone the repo at `BRANCH` and install pinned dependencies
If pip asks to restart the runtime: `Runtime -> Restart session`, then `Run all` again.

In [ ]:
import shutil
def run(cmd, **kw):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, **kw)

url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")      # Colab secret (key icon); needed only for a private repo
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if REPO_DIR.exists():
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"])
else:
    print("$ git clone --depth 1 -b", BRANCH, REPO_URL, REPO_DIR)   # token not printed
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO_DIR)], check=True)
GIT_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", GIT_COMMIT)
run(["pip", "install", "-q", "-r", PHASE1 / "requirements.txt"])

import importlib.metadata as md_, platform
VERSIONS = {}
for pkg in ["ultralytics", "litert-torch", "ai-edge-litert", "torch", "torchvision", "numpy",
            "opencv-python", "opencv-python-headless", "pillow", "matplotlib"]:
    try:
        VERSIONS[pkg] = md_.version(pkg)
    except md_.PackageNotFoundError:
        pass
VERSIONS.update(python=sys.version.split()[0], cuda=torch.version.cuda, platform=platform.platform())
for k, v in VERSIONS.items():
    print(f"{k:24s} {v}")
assert VERSIONS["ultralytics"] == "8.4.198", "unexpected ultralytics version; requirements.txt not applied"
sys.path.insert(0, str(PHASE1 / "scripts"))
sys.path.insert(0, str(PHASE1))

## 3. Steel data: the Phase 1.1 NEU 70/15/15 split (seed 42), asserted identical

In [ ]:
import json
run([sys.executable, PHASE1 / "scripts" / "prepare_data.py",
     "--raw-dir", PHASE1 / "data" / "raw", "--dataset-dir", PHASE1 / "data" / "dataset",
     "--seed", SEED, "--split", SPLIT])
CLS7_DIR = PHASE1 / "data" / "cls7"
run([sys.executable, PHASE1 / "scripts" / "build_cls_dataset.py",
     "--dataset-dir", PHASE1 / "data" / "dataset", "--out-dir", CLS7_DIR])

from build_cls_dataset import class_from_filename
from fusion import (FUSION_CONFIG, NEU_CLASSES, CLS7_CLASSES, NOT_STEEL, classifier_top1, fuse)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
def images(d):
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMG_EXT)
SPLITS = ("train", "val", "test")
STEEL = {sp: images(PHASE1 / "data" / "dataset" / "images" / sp) for sp in SPLITS}

MANIFEST = json.loads((PHASE1 / "results" / "phase1_2" / "neu_split_manifest_s42.json").read_text())
for sp in SPLITS:
    names = [p.name for p in STEEL[sp]]
    assert names == MANIFEST["splits"][sp]["files"], f"{sp}: steel split differs from the Phase 1.1 split"
    assert hashlib.sha256("\n".join(names).encode()).hexdigest() == MANIFEST["splits"][sp]["sha256_of_sorted_names"]
print("[CHECK] steel split identical to Phase 1.1:", {sp: len(v) for sp, v in STEEL.items()})

## 4. Download the not_steel sources (official URLs, see `DATA_SOURCES.md`)
About 2 GB. Archives are kept in `/content/ns_download` so a re-run does not download again.

In [ ]:
SOURCES = {
    "coco_val2017": {"url": "http://images.cocodataset.org/zips/val2017.zip",
                     "licence": "COCO terms of use; images: Flickr, per-image Creative Commons licences; annotations (not used) CC BY 4.0"},
    "dtd":          {"url": "https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz",
                     "licence": "DTD: research purposes only (Cimpoi et al., CVPR 2014)"},
    "places365_val_256": {"url": "https://data.csail.mit.edu/places/places365/val_256.tar",
                     "licence": "Places365: non-commercial research and educational purposes only (Zhou et al., TPAMI 2017)"},
    "places365_filelist": {"url": "https://data.csail.mit.edu/places/places365/filelist_places365-standard.tar",
                     "licence": "Places365 (as above)"},
}
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

for key, s in SOURCES.items():
    dst = NS_DL / Path(s["url"]).name
    if not dst.is_file():
        run(["curl", "-L", "--fail", "--retry", "3", "-sS", "-o", f"{dst}.part", s["url"]])
        Path(f"{dst}.part").rename(dst)
    s.update(file=dst.name, size_bytes=dst.stat().st_size, sha256=sha256_file(dst))
    out = NS_DL / key
    if not out.is_dir():
        out.mkdir()
        if dst.suffix == ".zip":
            run(["unzip", "-q", dst, "-d", out])
        else:
            run(["tar", "-xf", dst, "-C", out, "--no-same-owner"])
    print(f"{key:20s} {s['size_bytes']/1e6:8.1f} MB  sha256 {s['sha256'][:16]}...")

def find_dir(root, name):
    hits = sorted(p for p in Path(root).rglob(name) if p.is_dir())
    assert hits, f"{name} not found under {root}"
    return hits[0]
def find_file(root, name):
    hits = sorted(Path(root).rglob(name))
    assert hits, f"{name} not found under {root}"
    return hits[0]
COCO_DIR   = find_dir(NS_DL / "coco_val2017", "val2017")
DTD_IMAGES = find_dir(NS_DL / "dtd", "images")
PLACES_DIR = find_dir(NS_DL / "places365_val_256", "val_256")
PLACES_VAL = find_file(NS_DL / "places365_filelist", "places365_val.txt")
PLACES_CAT = find_file(NS_DL / "places365_filelist", "categories_places365.txt")
print("COCO", len(images(COCO_DIR)), "| DTD categories", len([d for d in DTD_IMAGES.iterdir() if d.is_dir()]),
      "| Places val", len(images(PLACES_DIR)))
assert len(images(COCO_DIR)) == 5000 and len(images(PLACES_DIR)) == 36500
assert len([d for d in DTD_IMAGES.iterdir() if d.is_dir()]) == 47

## 5. Build the not_steel class (grayscale / 200x200 crops, 70/15/15, zero overlap) and check the 7-class ImageFolder

In [ ]:
run([sys.executable, PHASE1 / "scripts" / "build_not_steel.py",
     "--coco-dir", COCO_DIR, "--dtd-dir", DTD_IMAGES, "--places-dir", PLACES_DIR,
     "--places-val-list", PLACES_VAL, "--places-categories", PLACES_CAT, "--out-root", CLS7_DIR,
     "--n-coco", N_COCO, "--n-dtd", N_DTD, "--n-places", N_PLACES,
     "--gray-frac", GRAY_FRAC, "--crop-frac", CROP_FRAC, "--seed", SEED])
NS_MANIFEST = json.loads((CLS7_DIR / "not_steel_manifest.json").read_text())

NS = {sp: images(CLS7_DIR / sp / NOT_STEEL) for sp in SPLITS}
want_ns = {"train": 0.70, "val": 0.15, "test": 0.15}
COUNTS = {}
for sp in SPLITS:
    folders = sorted(d.name for d in (CLS7_DIR / sp).iterdir() if d.is_dir())
    assert folders == sorted(CLS7_CLASSES), f"{sp}: {folders}"
    per = {c: len(images(CLS7_DIR / sp / c)) for c in CLS7_CLASSES}
    assert all(per[c] == {"train": 210, "val": 45, "test": 45}[sp] for c in NEU_CLASSES), per
    assert abs(per[NOT_STEEL] - want_ns[sp] * (N_COCO + N_DTD + N_PLACES)) <= 3, per
    assert sorted(p.name for c in NEU_CLASSES for p in images(CLS7_DIR / sp / c)) == [p.name for p in STEEL[sp]]
    COUNTS[sp] = per
names = {sp: {p.name for p in NS[sp]} for sp in SPLITS}
hashes = {sp: {r["sha256"] for r in NS_MANIFEST["images"] if r["split"] == sp} for sp in SPLITS}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not names[a] & names[b] and not hashes[a] & hashes[b], f"not_steel {a}/{b} overlap"
print(f"{'class':18s} " + " ".join(f"{sp:>6s}" for sp in SPLITS))
for c in CLS7_CLASSES:
    print(f"{c:18s} " + " ".join(f"{COUNTS[sp][c]:6d}" for sp in SPLITS))
print("not_steel: grayscale", NS_MANIFEST["grayscale"], "| cropped 200x200", NS_MANIFEST["cropped_200"],
      "| DTD categories", NS_MANIFEST["dtd_categories"], "| Places categories", NS_MANIFEST["places_categories"])
DATASET = {"split": SPLIT, "seed": SEED, "counts": COUNTS, "steel_split_identical_to_phase1_1": True,
           "not_steel": {k: v for k, v in NS_MANIFEST.items() if k != "images"}, "pairwise_overlap": 0}

import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, p in zip(axes.flat, NS["train"][::max(1, len(NS["train"]) // 24)]):
    ax.imshow(Image.open(p).convert("RGB")); ax.set_title(p.name.split("__")[0], fontsize=7); ax.axis("off")
plt.suptitle("not_steel train samples"); plt.tight_layout(); plt.savefig(RESULTS_DIR / "not_steel_samples.png", dpi=100); plt.show()

## 6. Phase 1.1 detector (unchanged), copied from Drive

In [ ]:
from export_child import available_precisions
from ultralytics import YOLO
P11_DET = P11_DRIVE / "models" / "detector"
DET_LOCAL = WORK / "p11_detector"
shutil.copytree(P11_DET, DET_LOCAL, dirs_exist_ok=True)
DET_IO = json.loads((DET_LOCAL / "model_io.json").read_text())
DET_PRECS = available_precisions(DET_IO)
DET_TFLITE = {p: DET_LOCAL / DET_IO["models"][p]["file"] for p in DET_PRECS}
DET_PT = DET_LOCAL / f"best_{P11_RUN_NAME}_detector.pt"
for f in [DET_PT, *DET_TFLITE.values()]:
    assert f.is_file(), f
# SHA-256 of the Phase 1.1 detector files (MODEL_IO.md -> Phase 1.1 model files)
P11_DET_SHA256 = {"best_p11_split701515_det320_cls224_s42_detector.pt": "4bdd364d2cc1e301fc4e5db918a21f2100840e901d1260f21dd7a0c1fb877c56",
                  "defect_yolov8n_320_fp32.tflite": "b453d1b99fe83a340c159ca44e99a9511cb3ee5b2dc2f813575aa2732f9262e0",
                  "defect_yolov8n_320_int8.tflite": "ceed1ecd20c6d8cb715b39cf7f2bc1cff238f4aac3f4fca6fb6278c547fb9f2f"}
for f in [DET_PT, *DET_TFLITE.values()]:
    assert sha256_file(f) == P11_DET_SHA256[f.name], f"{f.name}: not the Phase 1.1 detector file"
det_best = YOLO(str(DET_PT))
assert det_best.task == "detect" and [det_best.names[i] for i in sorted(det_best.names)] == NEU_CLASSES
P11_FUSION = json.loads((PHASE1 / "results" / "phase1_1" / "fusion_config.json").read_text())
assert FUSION_CONFIG["det_class_min_confidence"] == P11_FUSION["det_class_min_confidence"]
assert FUSION_CONFIG["cls_threshold"] == P11_FUSION["cls_threshold"] == 0.64
print("detector precisions:", DET_PRECS, "| thresholds (Phase 1.1, unchanged):", FUSION_CONFIG["det_class_min_confidence"])

## 7. Classifier v2: YOLOv8n-cls, 7 classes, imgsz 224, patience 20, seed 42
Ultralytics validates on `cls7/val` during training; `test` is not touched until step 11. ImageFolder sorts classes alphabetically (`not_steel` = index 2), so the trained head is then permuted to the contract order (6 NEU classes, then `not_steel`) and checked on real images.

In [ ]:
import time
import numpy as np
def epochs_in(run_dir):
    csv = Path(run_dir) / "results.csv"
    return len(csv.read_text().strip().splitlines()) - 1 if csv.is_file() else None

t0 = time.time()
cls_model = YOLO("yolov8n-cls.pt")   # ImageNet-pretrained starting point
cls_model.train(data=str(CLS7_DIR), epochs=CLS_EPOCHS, patience=CLS_PATIENCE, imgsz=CLS_IMGSZ, batch=CLS_BATCH,
                seed=SEED, deterministic=True, workers=2, device=DEVICE,
                project=str(WORK / "runs"), name=CLS_RUN, exist_ok=True, plots=True)
CLS_TRAIN_MIN = (time.time() - t0) / 60
CLS_TRAIN_DIR = Path(cls_model.trainer.save_dir)
CLS_PT_RAW = CLS_TRAIN_DIR / "weights" / "best.pt"
assert CLS_PT_RAW.is_file(), CLS_PT_RAW
CLS_EPOCHS_DONE = epochs_in(CLS_TRAIN_DIR)

from reorder_cls_classes import reorder_checkpoint
CLS_PT = WORK / "cls7_ordered" / f"best_{RUN_NAME}_classifier.pt"
CLS_PT.parent.mkdir(exist_ok=True)
REORDER = reorder_checkpoint(CLS_PT_RAW, CLS_PT)
raw_m, cls_best = YOLO(str(CLS_PT_RAW)), YOLO(str(CLS_PT))
assert cls_best.task == "classify" and [cls_best.names[i] for i in sorted(cls_best.names)] == CLS7_CLASSES
perm = REORDER["perm"]
probe = [*STEEL["val"][::27], *NS["val"][::30]]
REORDER_MAXDIFF = 0.0
for p in probe:
    a = raw_m.predict(str(p), imgsz=CLS_IMGSZ, verbose=False)[0].probs.data.cpu().numpy()
    b = cls_best.predict(str(p), imgsz=CLS_IMGSZ, verbose=False)[0].probs.data.cpu().numpy()
    REORDER_MAXDIFF = max(REORDER_MAXDIFF, float(np.abs(a[perm] - b).max()))
REORDER["check"] = {"images": len(probe), "max_abs_prob_diff": REORDER_MAXDIFF}
assert REORDER_MAXDIFF < 1e-4, REORDER_MAXDIFF
print(f"classifier: {CLS_PT} | epochs {CLS_EPOCHS_DONE} | {CLS_TRAIN_MIN:.1f} min | reorder check on {len(probe)} images: max diff {REORDER_MAXDIFF:.2e}")

## 8. Export FP32 TFLite (`--num-classes 7`, INT8 not attempted so it cannot hang)

In [ ]:
run([sys.executable, PHASE1 / "scripts" / "export_cls_tflite.py", "--weights", CLS_PT, "--data", CLS7_DIR,
     "--imgsz", CLS_IMGSZ, "--out-dir", EXPORT_CLS, "--num-classes", 7, "--precisions", "fp32",
     "--allow-skip-int8", "--timeout-min", EXPORT_TIMEOUT_MIN])
CLS_IO = json.loads((EXPORT_CLS / "model_io.json").read_text())
assert CLS_IO["classes"] == CLS7_CLASSES, CLS_IO["classes"]
assert (EXPORT_CLS / "labels.txt").read_text().split() == CLS7_CLASSES
assert CLS_IO["models"]["int8"]["status"] == "skipped"
CLS_PRECS = available_precisions(CLS_IO)
assert CLS_PRECS == ["fp32"], CLS_PRECS
c = CLS_IO["models"]["fp32"]
assert c["output_shape"] == [1, 7], c["output_shape"]
CLS_TFLITE = {"fp32": EXPORT_CLS / c["file"]}
print(f"classifier fp32: {c['size_bytes']/1e6:.2f} MB | input {c['input_shape']} {c['input_layout']} {c['input_dtype']} "
      f"| output {c['output_shape']} {c['output_dtype']} softmax={c['output_is_softmax']}")
print("classifier normalisation:", json.dumps(CLS_IO["preprocessing"], indent=2))

PIPELINES = {"pt": {"det": "pt", "cls": "pt", "label": ".pt pipeline: detector .pt + classifier v2 .pt"},
             "fp32": {"det": "fp32", "cls": "fp32", "label": "FP32 pipeline: detector FP32 + classifier v2 FP32"}}
if "int8" in DET_PRECS:
    PIPELINES["int8"] = {"det": "int8", "cls": "fp32", "label": "INT8 pipeline: detector INT8 + classifier v2 FP32"}
TFLITE_PIPES = [k for k in PIPELINES if k != "pt"]
for k, v in PIPELINES.items():
    print(f"{k:5s} -> {v['label']}")

## 9. Run every model on every val and test image (steel and not_steel)

In [ ]:
import copy
from tflite_utils import (load_interpreter, describe_io, assert_defect_model, run_tflite_on_image, describe_cls_io,
                          assert_classifier_model, run_cls_tflite, cls_preprocess)
det_interps = {p: load_interpreter(f) for p, f in DET_TFLITE.items()}
det_ios = {p: describe_io(it) for p, it in det_interps.items()}
for io_ in det_ios.values():
    assert_defect_model(io_)
cls_interps = {p: load_interpreter(f) for p, f in CLS_TFLITE.items()}
cls_ios = {p: describe_cls_io(it) for p, it in cls_interps.items()}
for io_ in cls_ios.values():
    assert_classifier_model(io_, 7)

def det_pt(path):
    r = det_best.predict(str(path), imgsz=DET_IMGSZ, conf=BASE_CONF, iou=NMS_IOU, verbose=False)[0]
    return [{"class_name": NEU_CLASSES[int(c)], "confidence": float(s), "bbox_xyxy": [float(v) for v in b]}
            for b, s, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())]
def cls_pt(path):
    return [float(v) for v in cls_best.predict(str(path), imgsz=CLS_IMGSZ, verbose=False)[0].probs.data.cpu().numpy()]
cls_torch = copy.deepcopy(cls_best.model).float().eval().to("cpu")
def cls_pt_refpre(path):
    x = cls_preprocess(np.asarray(Image.open(path).convert("RGB")), CLS_IMGSZ).transpose(2, 0, 1)[None]
    with torch.no_grad():
        out = cls_torch(torch.from_numpy(np.ascontiguousarray(x)))
    out = out[0] if isinstance(out, (list, tuple)) else out
    return out[0].cpu().numpy()

BASE, DETS, PROBS = {}, {}, {}
PREPROC_MAXDIFF = 0.0
for sp in ("val", "test"):
    imgs = [(p, class_from_filename(p.name)) for p in STEEL[sp]] + [(p, NOT_STEEL) for p in NS[sp]]
    BASE[sp], DETS[sp], PROBS[sp] = [], {m: [] for m in ["pt", *DET_PRECS]}, {m: [] for m in ["pt", *CLS_PRECS]}
    for img, true in imgs:
        w, h = Image.open(img).size
        BASE[sp].append({"image": img.name, "true_class": true, "width": w, "height": h})
        DETS[sp]["pt"].append(det_pt(img))
        p_pt = cls_pt(img)
        PROBS[sp]["pt"].append(p_pt)
        PREPROC_MAXDIFF = max(PREPROC_MAXDIFF, float(np.abs(np.array(p_pt) - cls_pt_refpre(img)).max()))
        for p in DET_PRECS:
            DETS[sp][p].append(run_tflite_on_image(det_interps[p], det_ios[p], img, NEU_CLASSES, BASE_CONF, NMS_IOU)[0])
        for p in CLS_PRECS:
            PROBS[sp][p].append([float(v) for v in run_cls_tflite(cls_interps[p], cls_ios[p], img)])
    print(f"{sp}: {len(STEEL[sp])} steel + {len(NS[sp])} not_steel images")

RECORDS = {sp: {k: [{**b, "detections": DETS[sp][v["det"]][i], "probs": PROBS[sp][v["cls"]][i]}
                    for i, b in enumerate(BASE[sp])] for k, v in PIPELINES.items()} for sp in BASE}
def steel(recs):
    return [r for r in recs if r["true_class"] != NOT_STEEL]
def notsteel(recs):
    return [r for r in recs if r["true_class"] == NOT_STEEL]
def probs_of(sp, m, want_ns):
    return [pr for b, pr in zip(BASE[sp], PROBS[sp][m]) if (b["true_class"] == NOT_STEEL) == want_ns]
print(f"classifier: max |prob(.pt) - prob(reference pre-processing)| = {PREPROC_MAXDIFF:.5f}")
if PREPROC_MAXDIFF > 0.02:
    raise AssertionError("tflite_utils.cls_preprocess does not match the .pt pre-processing")

## 10. Classifier `.pt` vs FP32 TFLite on ALL test images

In [ ]:
def top1(v):
    return int(np.argmax(v))
ref, oth = PROBS["test"]["pt"], PROBS["test"]["fp32"]
same = sum(top1(a) == top1(b) for a, b in zip(ref, oth))
maxdiff = max(float(np.abs(np.array(a) - np.array(b)).max()) for a, b in zip(ref, oth))
CLS_AGREEMENT = {"fp32": {"images": len(ref), "same_top1_as_pt": same, "rate": round(same / len(ref), 4),
                          "max_abs_prob_diff": round(maxdiff, 5),
                          "disagreements": [BASE["test"][i]["image"] for i, (a, b) in enumerate(zip(ref, oth)) if top1(a) != top1(b)]},
                 "int8": {"status": "skipped", "reason": CLS_IO["models"]["int8"]["reason"]}}
print(f"classifier fp32: same top-1 as .pt on {same}/{len(ref)} test images, max |dprob| {maxdiff:.4f}")

## 11. Thresholds on VAL only
`t_ns` from the FP32 classifier probabilities (the deployed model; the FP32 and INT8 pipelines share it): the most not_steel val images rejected with steel false rejection <= 1 %, plateau centre on ties. Then the classifier threshold is re-checked on the steel val images (TFLite pipelines summed, Phase 1.1 detector thresholds fixed): **0.64 is kept if it still reaches the best val score**, else the plateau centre is used.

In [ ]:
from fusion_eval import choose_not_steel_threshold, choose_cls_threshold, make_config, evaluate
NS_SEL = choose_not_steel_threshold([probs_of("val", "fp32", False)], [probs_of("val", "fp32", True)], MAX_FALSE_REJECT)
T_NS = NS_SEL["threshold"]
row = next(r for r in NS_SEL["sweep"] if r["threshold"] == T_NS)
print(f"t_ns = {T_NS} | val steel false rejects {row['steel_false_rejects']}/{row['steel_images']} "
      f"({row['steel_false_reject_rate']:.2%}) | val not_steel rejected {row['not_steel_rejected']}/{row['not_steel_images']} "
      f"({row['not_steel_rejection_rate']:.2%})")
for r in NS_SEL["sweep"][::5]:
    print(f"   t {r['threshold']:.2f}: steel FR {r['steel_false_rejects']:3d}  not_steel rejected {r['not_steel_rejected']:3d}  {'ok' if r['feasible'] else '-'}")

base = copy.deepcopy(FUSION_CONFIG); base["not_steel_threshold"] = T_NS
DET_THR = dict(FUSION_CONFIG["det_class_min_confidence"])          # Phase 1.1, unchanged
val_steel = [steel(RECORDS["val"][k]) for k in TFLITE_PIPES]
CLS_SEL = choose_cls_threshold(val_steel, DET_THR, base=base)
best = max(r["correct"] for r in CLS_SEL["sweep"])
at064 = next(r["correct"] for r in CLS_SEL["sweep"] if r["threshold"] == 0.64)
CLS_THR = 0.64 if at064 == best else CLS_SEL["threshold"]
print(f"classifier threshold re-check: 0.64 -> {at064}, best {best} (plateau centre {CLS_SEL['threshold']}) -> use {CLS_THR}"
      f" (of {sum(len(v) for v in val_steel)} steel val images, pipelines summed)")
FUSION_CFG = make_config(DET_THR, CLS_THR, base)

VAL_EVAL = {k: {"steel": evaluate(steel(RECORDS["val"][k]), FUSION_CFG),
                "not_steel": evaluate(notsteel(RECORDS["val"][k]), FUSION_CFG)} for k in PIPELINES}
for k, e in VAL_EVAL.items():
    s, n = e["steel"], e["not_steel"]
    print(f"  val [{PIPELINES[k]['label']}]: steel fused {s['correct']}/{s['images']}, steel false-rejected {s['retake']}, "
          f"not_steel rejected {n['retake']}/{n['images']}")
FUSION_CONFIG_OUT = {"selected_on": "val", "run_name": RUN_NAME, "git_commit": GIT_COMMIT,
                     "not_steel_threshold": T_NS, "max_false_reject": MAX_FALSE_REJECT,
                     "cls_threshold": CLS_THR, "cls_threshold_phase1_1": 0.64,
                     "cls_threshold_recheck": {"val_correct_at_0.64": at064, "val_best": best, "plateau_centre": CLS_SEL["threshold"]},
                     "det_class_min_confidence": DET_THR, "det_min_confidence": FUSION_CONFIG["det_min_confidence"],
                     "det_thresholds_source": f"Phase 1.1 {P11_RUN_NAME} (detector unchanged)",
                     "classifier_classes": CLS7_CLASSES,
                     "rule": "fusion.py: not_steel gate first; fusion_eval.choose_not_steel_threshold / choose_cls_threshold"}
(RESULTS_DIR / "fusion_config.json").write_text(json.dumps(FUSION_CONFIG_OUT, indent=2))

## 12. TEST report: 7x7 confusion, steel fused accuracy (Phase 1.1 detector), not_steel rejection, steel false rejection

In [ ]:
from fusion_eval import classifier_only_accuracy
def cls_confusion(m):
    cm = np.zeros((7, 7), int)
    for b, pr in zip(BASE["test"], PROBS["test"][m]):
        cm[CLS7_CLASSES.index(b["true_class"]), top1(pr)] += 1
    return cm
CLS_MODELS = ["pt", *CLS_PRECS]
CLS_CM = {m: cls_confusion(m) for m in CLS_MODELS}
CLS_TEST = {m: classifier_only_accuracy([{**b, "probs": PROBS["test"][m][i]} for i, b in enumerate(BASE["test"])], CLS7_CLASSES)
            for m in CLS_MODELS}
for m in CLS_MODELS:
    a = CLS_TEST[m]
    print(f"classifier v2 {m} top-1 (7 classes): {a['correct']}/{a['images']} ({a['accuracy']:.1%})")
    print("   true \\ pred " + " ".join(f"{c[:6]:>6s}" for c in CLS7_CLASSES))
    for i, c in enumerate(CLS7_CLASSES):
        print(f"   {c[:12]:12s} " + " ".join(f"{v:6d}" for v in CLS_CM[m][i]))

TEST = {}
for k in PIPELINES:
    s = evaluate(steel(RECORDS["test"][k]), FUSION_CFG)
    n = evaluate(notsteel(RECORDS["test"][k]), FUSION_CFG)
    missed = n["errors"]          # not_steel images that were not rejected
    TEST[k] = {"pipeline": PIPELINES[k],
               "steel_fused": s, "steel_false_rejects": s["retake"],
               "steel_false_reject_rate": round(s["retake"] / s["images"], 4),
               "not_steel": n, "not_steel_rejected": n["retake"],
               "not_steel_rejection_rate": round(n["retake"] / n["images"], 4),
               "not_steel_missed_outcome": {"no defect found": sum(e["final_class"] == FUSION_CFG["no_defect_label"] for e in missed),
                                            "defect alert": sum(e["final_class"] in NEU_CLASSES for e in missed)}}
    by_src = {}
    for b, pr in zip(BASE["test"], PROBS["test"][PIPELINES[k]["cls"]]):
        if b["true_class"] == NOT_STEEL:
            src = b["image"].split("__")[0]
            d = by_src.setdefault(src, {"images": 0, "rejected": 0})
            d["images"] += 1
            d["rejected"] += int(fuse([], classifier_top1(pr, CLS7_CLASSES), b["width"], b["height"], FUSION_CFG).decision == "RETAKE")
    TEST[k]["not_steel_rejection_by_source"] = by_src
    print(f"\nTEST [{PIPELINES[k]['label']}]")
    print(f"   steel fused image-level: {s['correct']}/{s['images']} ({s['accuracy']:.1%}) | decided by {s['decided_by']} | review {s['review']}")
    print(f"   steel false-rejected (RETAKE): {s['retake']}/{s['images']} ({TEST[k]['steel_false_reject_rate']:.2%})")
    print(f"   not_steel rejected (RETAKE):   {n['retake']}/{n['images']} ({TEST[k]['not_steel_rejection_rate']:.2%}) "
          f"| missed -> {TEST[k]['not_steel_missed_outcome']} | by source {by_src}")
    for err in s["errors"]:
        print(f"   STEEL ERROR {err['image']}: -> {err['final_class']} ({err['source']}) | {err['reason']}")
    if s["correct"] < s["images"]:
        print("!" * 100 + f"\n!!  steel fused accuracy {s['correct']}/{s['images']} is below the Phase 1.1 270/270\n" + "!" * 100)

In [ ]:
short = ["Cr", "In", "Pa", "PS", "RS", "Sc", "NS"]
fig, axes = plt.subplots(1, len(CLS_MODELS), figsize=(6 * len(CLS_MODELS), 5.2), squeeze=False)
for ax, m in zip(axes[0], CLS_MODELS):
    cm = CLS_CM[m]
    ax.imshow(np.log1p(cm), cmap="Blues")
    for i in range(7):
        for j in range(7):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 3 else "black", fontsize=8)
    ax.set_xticks(range(7), short); ax.set_yticks(range(7), short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"classifier v2 {m.upper()} (test): {CLS_TEST[m]['accuracy']:.1%}")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "cls7_confusion_test.png", dpi=120); plt.show()

ts = [r["threshold"] for r in NS_SEL["sweep"]]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ts, [r["not_steel_rejection_rate"] for r in NS_SEL["sweep"]], label="not_steel rejected (val)")
ax.plot(ts, [r["steel_false_reject_rate"] for r in NS_SEL["sweep"]], label="steel false-rejected (val)")
ax.axhline(MAX_FALSE_REJECT, ls=":", c="grey"); ax.axvline(T_NS, ls="--", c="k", label=f"t_ns = {T_NS}")
ax.set_xlabel("t_ns"); ax.set_ylabel("rate"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "not_steel_threshold_val.png", dpi=120); plt.show()

## 13. Save to the NEW Drive run folder `p12_cls7_notsteel_s42`
The Phase 1 and Phase 1.1 folders are only read; both are checked unchanged.

In [ ]:
from datetime import datetime, timezone
RESULTS = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT, "branch": BRANCH, "gpu": GPU_NAME, "versions": VERSIONS, "run_name": RUN_NAME,
    "config": {"seed": SEED, "split": SPLIT, "detector": f"Phase 1.1 {P11_RUN_NAME}, unchanged",
               "classifier": {"base": "yolov8n-cls.pt (ImageNet)", "classes": CLS7_CLASSES, "imgsz": CLS_IMGSZ,
                              "epochs": CLS_EPOCHS, "patience": CLS_PATIENCE, "batch": CLS_BATCH},
               "not_steel": {"n_coco": N_COCO, "n_dtd": N_DTD, "n_places": N_PLACES, "gray_frac": GRAY_FRAC, "crop_frac": CROP_FRAC},
               "max_false_reject": MAX_FALSE_REJECT, "sweep_decode": {"base_conf": BASE_CONF, "nms_iou": NMS_IOU}},
    "data_sources": SOURCES, "dataset": DATASET,
    "training": {"classifier_minutes": round(CLS_TRAIN_MIN, 1), "classifier_epochs_completed": CLS_EPOCHS_DONE,
                 "reorder": REORDER},
    "pipelines": PIPELINES,
    "model_io": {"detector": DET_IO, "classifier": CLS_IO},
    "classifier_preprocessing_parity_max_abs_prob_diff": PREPROC_MAXDIFF,
    "threshold_selection_val": {"not_steel": NS_SEL, "classifier": CLS_SEL, "fusion_config": FUSION_CONFIG_OUT, "val": VAL_EVAL},
    "test": {"classifier_top1_7class": CLS_TEST, "classifier_confusion_7x7": {m: cm.tolist() for m, cm in CLS_CM.items()},
             "confusion_order": CLS7_CLASSES, "classifier_agreement_vs_pt": CLS_AGREEMENT, "pipelines": TEST,
             "note": "NEU-DET has no defect-free steel images; not_steel = COCO/DTD/Places365, not factory scenes."},
}
(RESULTS_DIR / "phase1_2_results.json").write_text(json.dumps(RESULTS, indent=2))
shutil.copy2(CLS7_DIR / "not_steel_manifest.json", RESULTS_DIR / "not_steel_manifest.json")
for src in [*CLS_TRAIN_DIR.glob("*.png"), CLS_TRAIN_DIR / "results.csv"]:
    if Path(src).is_file():
        shutil.copy2(src, RESULTS_DIR / f"cls_{Path(src).name}")

MODELS_OUT = WORK / "models" / "classifier"
MODELS_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(CLS_PT, MODELS_OUT / CLS_PT.name)
shutil.copy2(CLS_PT_RAW, MODELS_OUT / f"best_{RUN_NAME}_classifier_alphabetical_order.pt")
for f in [*EXPORT_CLS.glob("*.tflite"), EXPORT_CLS / "labels.txt", EXPORT_CLS / "model_io.json"]:
    shutil.copy2(f, MODELS_OUT / f.name)

assert not DRIVE_DEST.exists(), DRIVE_DEST
shutil.copytree(RESULTS_DIR, DRIVE_DEST / "results")
shutil.copytree(WORK / "models", DRIVE_DEST / "models")
assert tree_fingerprint(OLD_DRIVE) == OLD_FINGERPRINT, "the Phase 1 Drive folder changed!"
assert tree_fingerprint(P11_DRIVE) == P11_FINGERPRINT, "the Phase 1.1 Drive folder changed!"
print("Saved to Drive:", DRIVE_DEST, "| Phase 1 and Phase 1.1 folders unchanged")
for p in sorted((WORK / "models").rglob("*")) + sorted(RESULTS_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.relative_to(WORK)}  ({p.stat().st_size/1e6:.2f} MB)")

## 14. After the run
1. Copy `<Drive>/p12_cls7_notsteel_s42/results/` into `phase1-vision/results/phase1_2/` and commit it
   (JSON + PNGs only; never `.pt` / `.tflite`).
2. Put `not_steel_threshold` (and `cls_threshold` if it changed) from `fusion_config.json` into
   `FUSION_CONFIG` in `phase1-vision/fusion.py`, and pin them with a test like `TunedConfigTests`.
3. Fill the classifier v2 TODOs in `phase1-vision/MODEL_IO.md` from `models/classifier/model_io.json`
   and the Phase 1.2 tracker in `PROJECT_CONTEXT.md` with the TEST numbers.